# Phase 6 DAEAC Paper-Faithful Kaggle Driver

This notebook is a thin driver. It calls repository scripts instead of duplicating training logic.

In [ ]:
from pathlib import Path
import glob, os, shutil, subprocess, sys, textwrap
os.environ['PYTHONUNBUFFERED'] = '1'  # stream subprocess logs/tqdm live on Kaggle

# Edit this to your GitHub repo URL before running on Kaggle.
REPO_URL = 'https://github.com/YOUR_ORG/Best-thesis-in-council.git'
BRANCH = 'main'
CONFIG = 'configs/phase6_daeac_paper.yaml'
DUAL_HEAD_CONFIG = 'configs/phase6_daeac_paper_dualhead.yaml'
USE_DUAL_HEAD = True
ACTIVE_CONFIG = DUAL_HEAD_CONFIG if USE_DUAL_HEAD else CONFIG
PAIRS = ['ds1_ds2', 'ds1_incart', 'ds1_svdb', 'mitbih_incart', 'mitbih_svdb']
EVAL_DATASET = 'target'

WORK = Path('/kaggle/working')
REPO = WORK / 'Best-thesis-in-council'
ECG = REPO / 'ecg_thesis'
print('Kaggle working:', WORK.exists())
print('Repo path:', ECG)

## 1. Clone Repo From GitHub

Edit `REPO_URL` in the first cell, then run this cell. If the repo already exists in `/kaggle/working`, the cell reuses it.

In [ ]:
if not ECG.exists():
    assert 'YOUR_ORG' not in REPO_URL, 'Edit REPO_URL before cloning.'
    !git clone --branch {BRANCH} {REPO_URL} /kaggle/working/Best-thesis-in-council
else:
    print('Repo already exists:', ECG)
assert ECG.exists(), f'Missing repo at {ECG}. Clone or upload it first.'
os.chdir(ECG)
print(Path.cwd())
!git status --short || true

## 2. Install Dependencies

In [ ]:
!pip install -q -r requirements.txt

## 3. Copy Preprocessed DAEAC Data

This notebook expects the Kaggle dataset structure shown by the user: a folder named `daeac` containing files such as `mitdb_ds1_daeac.npz`, `mitdb_ds2_daeac.npz`, and `mitdb_ds2_first5_unlabeled_daeac.npz`.

In [ ]:
candidate_dirs = [p.parent for p in Path('/kaggle/input').glob('**/record_split_manifest.json')]
print('Candidate daeac data dirs:')
for p in candidate_dirs:
    print(' -', p)

assert len(candidate_dirs) == 1, f'Expected one record-split bundle, found {candidate_dirs}'
KAGGLE_DATASET_DIR = candidate_dirs[0]
print('Using data dir:', KAGGLE_DATASET_DIR)

DEST = Path('data/processed/phase6_daeac_record_splits')
DEST.mkdir(parents=True, exist_ok=True)
print('Available npz files under /kaggle/input:')
for p in sorted(Path('/kaggle/input').glob('**/*.npz')):
    print(' -', p)

if KAGGLE_DATASET_DIR.exists():
    for p in KAGGLE_DATASET_DIR.glob('*.npz'):
        shutil.copy2(p, DEST / p.name)
        print('copied', p.name)
else:
    raise FileNotFoundError('Could not find the daeac dataset folder. Edit KAGGLE_DATASET_DIR manually.')

print('Copied files:')
!ls -lh data/processed/phase6_daeac_record_splits

DS1_BASE_CKPT = Path('/kaggle/input/models/thanhvu2357/daeac-base-best/pytorch/default/1/daeac_base_best.pt')
MITBIH_BASE_CKPT = Path('/kaggle/input/models/noname259/daeac-base-mitbih-best/pytorch/default/1/daeac_base_mitbih_best.pt')
if not DS1_BASE_CKPT.exists():
    matches = glob.glob('/kaggle/input/**/daeac_base_best.pt', recursive=True)
    assert matches, 'Missing attached DS1 base checkpoint daeac_base_best.pt'
    DS1_BASE_CKPT = Path(matches[0])
if not MITBIH_BASE_CKPT.exists():
    matches = glob.glob('/kaggle/input/**/daeac_base_mitbih_best.pt', recursive=True)
    assert len(matches) == 1, 'Missing attached MITBIH checkpoint daeac_base_mitbih_best.pt'
    MITBIH_BASE_CKPT = Path(matches[0])
print('DS1_BASE_CKPT =', DS1_BASE_CKPT)
print('MITBIH_BASE_CKPT =', MITBIH_BASE_CKPT, '(not used by the DS1->DS2 run)')

## 4. Validate Repo and Data

In [ ]:
!python scripts/check_repo.py
subprocess.run([sys.executable, 'scripts/daeac/03_validate_kaggle_bundle.py', '--bundle', str(KAGGLE_DATASET_DIR)], check=True)
subprocess.run([sys.executable, 'scripts/phase6_daeac_paper/00_validate_data.py', '--config', ACTIVE_CONFIG], check=True)

## 5. Smoke Run

Smoke chỉ kiểm tra base-training bằng prefix riêng. Không chạy UDA từ base 1 epoch vì các threshold paper (N>0.999; S/V/F>0.99) có thể hợp lệ nhưng chưa chọn được mẫu nào.

In [ ]:
RUN_SMOKE = False
SMOKE_EPOCHS = '1'
SMOKE_SAMPLES = '512'
if RUN_SMOKE:
    subprocess.run([
        sys.executable, '-u', 'scripts/phase6_daeac_paper/01_train_base.py',
        '--config', ACTIVE_CONFIG, '--epochs', SMOKE_EPOCHS,
        '--max-source-samples', SMOKE_SAMPLES, '--max-val-samples', SMOKE_SAMPLES,
        '--checkpoint-prefix', 'daeac_base_smoke',
    ], check=True)
    subprocess.run([
        sys.executable, '-u', 'scripts/phase6_daeac_paper/02_adapt_uda.py',
        '--config', ACTIVE_CONFIG, '--domain-pair', 'ds1_ds2', '--init-checkpoint', str(DS1_BASE_CKPT),
        '--epochs', SMOKE_EPOCHS,
        '--max-source-samples', SMOKE_SAMPLES, '--max-target-samples', SMOKE_SAMPLES, '--max-val-samples', SMOKE_SAMPLES,
        '--checkpoint-prefix', 'daeac_paper_dualhead_ds1_ds2_smoke' if USE_DUAL_HEAD else 'daeac_paper_ds1_ds2_smoke',
    ], check=True)

## 5A. Dual-Head Smoke Run

In [ ]:
RUN_DUAL_HEAD_SMOKE = False
if RUN_DUAL_HEAD_SMOKE:
    subprocess.run([
        sys.executable, '-u', 'scripts/phase6_daeac_paper/02_adapt_uda.py',
        '--config', ACTIVE_CONFIG,
        '--domain-pair', 'ds1_ds2',
        '--init-checkpoint', str(DS1_BASE_CKPT),
        '--epochs', '1',
        '--max-source-samples', '512',
        '--max-target-samples', '512',
        '--max-val-samples', '512',
        '--checkpoint-prefix', 'daeac_paper_dualhead_ds1_ds2_smoke',
    ], check=True)

## 6. Full Run

Mặc định bỏ qua pretrain và adaptation 300 epochs trực tiếp từ DS1 checkpoint đã attach. Toàn bộ DS1 có nhãn được dùng làm source adaptation; không tách validation.

In [ ]:
for pair in PAIRS:
    base_ckpt = MITBIH_BASE_CKPT if pair.startswith('mitbih_') else DS1_BASE_CKPT
    if Path(f'outputs/phase6_daeac_paper_{pair}/checkpoints/daeac_paper_{pair}_best.pt').exists(): print('resume: skip', pair); continue
    subprocess.run([sys.executable, '-u', 'scripts/phase6_daeac_paper/02_adapt_uda.py', '--config', ACTIVE_CONFIG, '--domain-pair', pair, '--init-checkpoint', str(base_ckpt)], check=True)

## 7. Evaluate and Report

In [ ]:
for pair in PAIRS:
    checkpoint = f'outputs/phase6_daeac_paper_{pair}/checkpoints/daeac_paper_{pair}_best.pt'
    subprocess.run([sys.executable, '-u', 'scripts/phase6_daeac_paper/03_eval.py', '--config', ACTIVE_CONFIG, '--domain-pair', pair, '--checkpoint', checkpoint, '--method-name', f'daeac_paper_{pair}_best_v', '--dataset', EVAL_DATASET], check=True)

## 8. Zip Outputs

In [ ]:
!zip -r /kaggle/working/phase6_daeac_paper_outputs.zip outputs/phase6_daeac_paper_*